# AIAIC forecast training (Google Colab)

This notebook reproduces the local forecast run for Onion, Soyabean, Tur, and Wheat at 7- and 30-day horizons.

**Upload this dataset:** `backend_files/agmarknet_pilot_mh_mp_2001_2026.csv.gz` (about 38 MB).
It contains the Agmarknet pilot data through 2026-08-26. `mandi_identity.csv` is not required for this
training command because the dataset already includes the canonical `mandi` and `crop` columns.
GPU is optional; this LightGBM run is CPU-capable and uses the same deterministic seed as the local run.

In [ ]:
!pip -q install lightgbm pandas==2.2.2 numpy==1.26.4 scikit-learn==1.5.0 joblib

In [ ]:
from google.colab import files
uploaded = files.upload()
dataset_name = "agmarknet_pilot_mh_mp_2001_2026.csv.gz"
if dataset_name not in uploaded:
    raise FileNotFoundError(f"Upload {dataset_name}; received {list(uploaded)}")


In [ ]:
!git clone --depth 1 https://github.com/blackholeinfiverse37/aiaic-ml-foundation.git repo
!mkdir -p repo/backend_files
!cp agmarknet_pilot_mh_mp_2001_2026.csv.gz repo/backend_files/
%cd repo

In [ ]:
!mkdir -p models/forecast_colab
!python -m src.forecast.train \
  --input backend_files/agmarknet_pilot_mh_mp_2001_2026.csv.gz \
  --market-column mandi \
  --commodity-column crop \
  --crops onion soybean tur wheat \
  --horizons 7 30 \
  --holdout-days 180 \
  --step-days 7 \
  --seed 42 \
  --models-dir models/forecast_colab \
  --source "Agmarknet via data.gov.in, AIAIC pilot export 2026-10-02"

In [ ]:
import glob, json, pathlib, shutil

meta_path = sorted(pathlib.Path("models/forecast_colab").glob("forecast_*.json"))[-1]
meta = json.loads(meta_path.read_text())
print(json.dumps({
    "model_file": meta["model_file"],
    "trained_through": meta["trained_through"],
    "backtest_cutoff": meta["backtest_cutoff"],
    "backtest_overall": meta["backtest"]["overall"],
    "usable_groups": [g for g in meta["backtest"]["groups"] if g["usable"]],
}, indent=2))

release_dir = pathlib.Path("colab_forecast_bundle")
if release_dir.exists():
    shutil.rmtree(release_dir)
release_dir.mkdir()
for name in (meta_path.name, meta["model_file"], meta["history_file"]):
    shutil.copy2(pathlib.Path("models/forecast_colab") / name, release_dir / name)
(release_dir / "METRICS.json").write_text(json.dumps(meta["backtest"], indent=2) + "\n")
shutil.make_archive("aiaic_forecast_colab_bundle", "zip", release_dir)
print("Created aiaic_forecast_colab_bundle.zip")

In [ ]:
from google.colab import files
files.download("aiaic_forecast_colab_bundle.zip")